# 2b · Which metric drives IQB, and is each one weighted as intended?

**Research question (RQ 2b):** *Which of the four metrics (download, upload, latency, packet loss) actually makes a difference to a country's IQB score at P25, P50, P75 and P95? And does each metric, or combination of metrics, carry the weight the framework intends?*

IQB is a fixed formula of the four metrics, so we ask whether each metric matters in practice as much as its weight says.

This is the standard diagnostic for composite indicators: compare nominal weights with measured importance (Paruolo, Saisana & Saltelli 2013, *JRSS A*; Becker et al. 2017, *Ecological Indicators*). A metric's influence depends not only on its weight but also on how often its thresholds separate countries, and on how correlated it is with the other metrics.

## How much does each metric contribute to the IQB score?

The configuration gives each of the ten (metric, threshold) tests a weight. A metric's nominal weight is the sum over its tests: its maximum possible contribution to the score.

IQB at percentile q is the sum of four parts, one per metric: the weights of that metric's tests that pass. 

Because the score is a plain sum, the differences in IQB between country-months can be split exactly into four components, one per metric, that add up to the whole. We call each metric's piece its effective share: the fraction of the differences in IQB that comes from that metric:

$$\operatorname{Var}(\text{IQB}) = \sum_m \operatorname{Cov}(c_m, \text{IQB}), \qquad \text{effective share}_m = \frac{\operatorname{Cov}(c_m, \text{IQB})}{\operatorname{Var}(\text{IQB})}$$

A metric has a large effective share when its tests pass for some countries and fail for others, and when that pattern lines up with the overall score. A metric whose tests pass (or fail) for everyone contributes a constant: its weight then shifts every score equally but never separates two countries.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from scipy.stats import spearmanr
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.metrics import r2_score
from sklearn.model_selection import GroupKFold

import iqb_tools as T

T.set_style()
LEVELS = (25, 50, 75, 95)
METRIC_LABEL = {"download": "download", "upload": "upload", "latency": "latency", "loss": "packet loss"}
METRIC_COLOR = dict(zip(T.METRICS, T.SERIES[:4]))

df = T.load_level("country")
df = df[T.binding_tests(df) >= T.MIN_TESTS].reset_index(drop=True)
cfg = T.config_table()
W = T.effective_test_weights(cfg)
tests = T.distinct_tests(cfg).set_index("key")

print(f"{len(df):,} country-months from {df.country_code.nunique()} countries with ≥ {T.MIN_TESTS:,} tests in both directions")

metric_of = tests.metric
nominal = W.groupby(metric_of.reindex(W.index)).sum().reindex(list(T.METRICS))
nominal_share = nominal / nominal.sum()
pd.DataFrame(
    {
        "tests": [
            ", ".join(k for k in W.index if metric_of[k] == m) for m in T.METRICS
        ],
        "nominal weight": nominal.round(3),
    },
    index=list(T.METRICS),
)

The above table gives the sample for this notebook: every country-month with at least 1,153 tests in both directions (the minimum from 1a), 3,834 of them from 186 countries over 2024 and 2025. 

The table lists, for each metric, its tests (metric and threshold; *download>10* is download above 10 Mbit/s) and its nominal weight: the share of the IQB score its tests carry when they all pass, so the four weights add up to 1. Packet loss carries the most weight (0.283) and upload the least (0.192).

In [ ]:
def contributions(d, q):
    """Per-metric contribution to IQB at percentile q (sums to the IQB score)."""
    c = pd.DataFrame(0.0, index=d.index, columns=list(T.METRICS))
    for k, w in W.items():
        m, t = tests.loc[k, "metric"], tests.loc[k, "threshold"]
        v = d[f"{m}_p{q}"]
        c[m] += w * (v > t if T.HIGHER_IS_BETTER[m] else v < t)
    return c


def effective_shares(c):
    y = c.sum(axis=1)
    return c.apply(lambda col: np.cov(col, y)[0, 1]) / y.var()


contrib = {q: contributions(df, q) for q in LEVELS}
eff = pd.DataFrame({f"P{q}": effective_shares(contrib[q]) for q in LEVELS})
eff.insert(0, "nominal", nominal_share)
always = pd.DataFrame(
    {
        f"P{q}": [(contrib[q][m] == nominal[m]).mean() for m in T.METRICS]
        for q in LEVELS
    },
    index=list(T.METRICS),
)
print("Effective share of the variance of IQB across country-months:")
eff.round(2)

Each value is a metric's share of the differences in IQB between country-months. 

The first column ``nominal`` is its nominal share from the weights, and the others are its effective share when IQB is computed at P25, P50, P75 or P95. Each column adds up to 1. A metric's effective share is high when its tests pass for some country-months and fail for others, in line with the overall score.

The effective shares move far from the nominal ones as the percentile changes. Download's share falls steadily from 42% at P25 to 17% at P95, while upload's rises from 8% to 32%, and latency's is highest at the two ends (39% at P25 and 50% at P95). Packet loss swings from 10% at P25 to 34% at P50, then down to 11% at P75 and 0% at P95.

In [ ]:
print("Share of country-months in which ALL of a metric's tests pass (its weight is then a constant):")
always.round(2)

Each value is the share of country-months in which *all* of a metric's tests pass at that percentile, so that the metric adds its full weight to every one of those scores. 

Where the share is close to 1, the metric adds the same amount to almost every score and cannot separate countries, which is why its effective share above is close to zero. (The opposite case, where all of a metric's tests fail, has the same effect. That is close to what happens to packet loss at P25, whose tests fail in almost every country-month, and partly to upload, whose tests all fail in most of them.)

The shares rise with the percentile for every metric, because a higher percentile describes a better performance in that metric. Latency is the hardest metric to pass in full (38% at P95), presumably because one of its tests requires latency below 10 ms.

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 3.6))
cols = ["nominal"] + [f"P{q}" for q in LEVELS]
x = np.arange(len(cols))
slot, bar = 0.12, 0.11
for k, m in enumerate(T.METRICS):
    ax.bar(x + (k - 1.5) * slot, eff.loc[m, cols], bar, color=METRIC_COLOR[m], label=METRIC_LABEL[m])
ax.axvline(0.5, color=T.AXIS, lw=1)
ax.set_xticks(x, ["nominal weight"] + [f"effective at P{q}" for q in LEVELS])
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
ax.grid(axis="x", visible=False)
ax.set_ylabel("share")
ax.set_title("Intended vs actual influence of each metric on IQB")
ax.legend(ncol=4, loc="upper left", fontsize=9)
T.savefig(fig, "2b_nominal_vs_effective")

The figure shows the first table as bars. 

Each group of four bars is one column of the table: on the left, separated by a vertical line, the nominal weights, then the effective shares at P25, P50, P75 and P95. 

The x-axis lists these five groups, the y-axis is the share (0–50%), and the colors are the four metrics. 

If every metric had its intended influence, each group would look like the nominal one.

The tables and figure show that metrics do not carry its intended weight at every percentile.

At **P50** the effective shares are fairly close to the nominal ones: download 31%, packet loss 34%, latency 20% and upload 15%, against 27, 28, 26 and 19%. Latency counts somewhat less than intended, and loss somewhat more.

At **P25** download and latency explain four fifths of the differences between countries. Upload and loss barely matter (8% and 10%): their tests almost never pass at P25.

At **P95** packet loss has **no influence at all**. All its tests pass in every country-month, so its 28% weight adds a constant 0.28 to every score, which is part of why P95 scores crowd at the top of the scale. Download, which passes in 98% of country-months, explains only 17%, and latency explains half.

The weights are fixed, but which thresholds separate countries depends on the percentile, so the same weights produce very different effective weightings.

## Does a random forest agree?

We train a random forest to predict IQB at each percentile from the four metric values at that percentile, using country-grouped 5-fold cross-validation: a country's months are never split between training and test, so the forest cannot memorize countries. 

The forest has four features: the download, upload, latency and packet-loss values at a given percentile. Its target is the IQB score at that percentile.
All three measures are built on held-out R²: the share of the variation in IQB that the forest predicts correctly for countries it was not trained on. A value of 1 means perfect prediction. 


For each metric we report three measures:

**Permutation importance**: Train the forest once with all four features. Then, on the held-out countries, shuffle one feature's values and see how much R² drops. How much does this trained forest rely on the feature?

**Drop-column importance**: Retrain the forest without the feature and see how much R² drops compared with the full forest. What does the feature add that the other features can't replace?

**Single-metric R²**: Train a forest on only that feature. How much can the feature predict on its own, including what it shares with other features?

In [ ]:
FOREST = dict(n_estimators=300, min_samples_leaf=5, n_jobs=-1, random_state=11)
folds = list(GroupKFold(n_splits=5).split(df, groups=df.country_code))


def oof_r2(X, y, model_fn):
    pred = np.zeros(len(y))
    for tr, te in folds:
        pred[te] = model_fn().fit(X.iloc[tr], y.iloc[tr]).predict(X.iloc[te])
    return r2_score(y, pred)


rf_rows = []
for q in LEVELS:
    X = df[[f"{m}_p{q}" for m in T.METRICS]].set_axis(list(T.METRICS), axis=1)
    y = contrib[q].sum(axis=1)
    r2_full = oof_r2(X, y, lambda: RandomForestRegressor(**FOREST))
    perm = pd.Series(0.0, index=list(T.METRICS))
    for tr, te in folds:
        f = RandomForestRegressor(**FOREST).fit(X.iloc[tr], y.iloc[tr])
        pi = permutation_importance(
            f, X.iloc[te], y.iloc[te], n_repeats=5, random_state=0, scoring="r2"
        )
        perm += pi.importances_mean / len(folds)
    for m in T.METRICS:
        rf_rows.append(
            {
                "level": f"P{q}",
                "metric": m,
                "R² full forest": r2_full,
                "permutation importance": perm[m],
                "drop-column importance": r2_full
                - oof_r2(X.drop(columns=m), y, lambda: RandomForestRegressor(**FOREST)),
                "single-metric R²": oof_r2(
                    X[[m]], y, lambda: RandomForestRegressor(**FOREST)
                ),
            }
        )
rf = pd.DataFrame(rf_rows)
print(
    "Held-out R² of the full forest:",
    rf.groupby("level")["R² full forest"].first().round(3).to_dict(),
)
rf.pivot(
    index="metric",
    columns="level",
    values=["permutation importance", "drop-column importance", "single-metric R²"],
).reindex(list(T.METRICS)).round(3)


The table gives the three importance measures defined above for each metric (rows) at each percentile (columns), all on the R² scale:

| Column group | Meaning |
|---|---|
| *permutation importance* | How much held-out R² drops when the metric's values are shuffled. |
| *drop-column importance* | How much held-out R² drops when the forest is retrained without the metric: what the other metrics cannot replace. |
| *single-metric R²* | The held-out R² of a forest that sees only this metric. |

For example, at P95 shuffling latency costs 0.795 of R², removing it costs 0.291, and latency alone predicts 57% of IQB. 

The three measures differ because the metrics are correlated: single-metric R² is far larger than drop-column importance in every cell, so most of what one metric explains, the others could explain too.

In [ ]:
# Does the forest agree with the exact decomposition?
perm_share = rf.pivot(
    index="metric", columns="level", values="permutation importance"
).clip(lower=0)
perm_share = (perm_share / perm_share.sum()).reindex(list(T.METRICS))
agree = {
    lv: round(float(spearmanr(perm_share[lv], eff[lv]).statistic), 2)
    for lv in perm_share.columns
}
print("Rank agreement (Spearman) between forest and exact importances:", agree)
pd.concat(
    {
        "forest (permutation, normalized)": perm_share,
        "exact effective share": eff[perm_share.columns],
    },
    axis=1,
).round(2)


To compare the forest with the exact decomposition, the permutation importances are rescaled so that each percentile's four values add up to 1 (left half of the table) and set next to the exact effective shares from the first table (right half). 

The printed line gives the Spearman rank correlation between the two at each percentile. 1.0 means the two methods rank the four metrics in the same order. 

They agree fully at P25 and P95, nearly at P50 (0.8) and least at P75 (0.4), where the forest gives download 46% against an exact 28%, and upload 17% against 30%.

At P75 it overstates download and understates upload, a known bias of permutation importance when features are correlated: the effect is credited to whichever correlated metric the trees split on.

Drop-column importances are small, under 0.1 in 11 of the 16 cases. The least replaceable are latency at P25, P75 and P95 (0.19, 0.14 and 0.29) and packet loss at P50 (0.16). The metric that makes a difference in the computation of the score depends on the percentile: download and latency at P25, loss and download at P50, latency and upload at P95.

### What does each metric do to the score?

In [ ]:
q = 50
X = df[[f"{m}_p{q}" for m in T.METRICS]].set_axis(list(T.METRICS), axis=1)
y = contrib[q].sum(axis=1)
forest = RandomForestRegressor(**FOREST).fit(X, y)
fig, axes = plt.subplots(1, 4, figsize=(14, 3.3), sharey=True)
for ax, m in zip(axes, T.METRICS):
    lo, hi = np.percentile(X[m], [2, 98])
    grid = np.geomspace(max(lo, 1e-4), hi, 80) if m != "loss" else np.linspace(lo, hi, 80)
    curve = []
    for v in grid:  # partial dependence: set the metric to v everywhere, average the predictions
        Xv = X.copy()
        Xv[m] = v
        curve.append(forest.predict(Xv).mean())
    ax.plot(grid, curve, color=METRIC_COLOR[m])
    for t in sorted(set(tests[tests.metric == m].threshold)):
        ax.axvline(t, color=T.AXIS, lw=0.8)
    if m in ("download", "upload", "latency"):
        ax.set_xscale("log")
    ax.set_title(METRIC_LABEL[m])
    ax.set_xlabel({"download": "Mbit/s", "upload": "Mbit/s", "latency": "ms", "loss": "loss rate"}[m])
axes[0].set_ylabel("IQB at P50 (partial dependence)")
fig.suptitle("How IQB at P50 responds to each metric, while holding the others at their observed values",
             x=0.01, ha="left", fontweight="bold")
fig.tight_layout()
T.savefig(fig, "2b_partial_dependence_p50")

Each panel shows one metric. Within each panel, vertical lines represent thresholds within that metric. 

The x-axis is the metric's value at P50, in Mbit/s for download and upload and in ms for latency (all on log scales), and as a loss rate for packet loss. 

The y-axis is the forest's average predicted IQB at P50 when every country-month is given that value of the metric and keeps its own values of the other three (the metric's *partial dependence*). 

A flat stretch means the metric makes no difference there, and a step means that crossing that value changes the score.

The figure shows steps at the thresholds: IQB responds to a metric only where it crosses a threshold, and between thresholds, improving a metric changes nothing in regards to the score. For example, the IQB download thresholds are 10 and 25 Mbit/s, so a country can raise its median download from 30 to 300 Mbit/s with no effect on IQB.

The biggest steps are at 10 and 25 Mbit/s download (from about 0.31 to 0.47, then to 0.63), 25 Mbit/s upload (about 0.54 to 0.62), 100 ms latency (about 0.57 to 0.40) and 1% packet loss (about 0.60 to 0.39). The 5 Mbit/s upload and 10 ms latency thresholds barely move the curve as can be seen in the panels. 

## Do combinations of metrics matter?

If IQB rewarded combinations (e.g., a video call needs low latency and sufficient upload speed), a model with interactions would predict it better than an additive one. We therefore compare the held-out R² of gradient-boosted trees with and without interactions allowed (`interaction_cst` makes the model a sum of one-metric functions).

We repeat the comparison for a non-compensatory variant of IQB, in which a use case counts only if *all* of its requirements pass. That is the "weakest link" logic used for the framework's classes in 2a.

In [ ]:
def additive_gap(X, y):
    full = oof_r2(
        X,
        y,
        lambda: HistGradientBoostingRegressor(
            max_iter=400, learning_rate=0.05, random_state=0
        ),
    )
    add = oof_r2(
        X,
        y,
        lambda: HistGradientBoostingRegressor(
            max_iter=400,
            learning_rate=0.05,
            random_state=0,
            interaction_cst=[[0], [1], [2], [3]],
        ),
    )
    return full, add


def weakest_link_iqb(d, q):
    """Mean over use cases of 1{every requirement passes} (non-compensatory)."""
    total = np.zeros(len(d))
    for uc, g in cfg.groupby("use_case"):
        ok = np.ones(len(d), dtype=bool)
        for r in g.itertuples():
            v = d[f"{r.metric}_p{q}"].to_numpy()
            ok &= v > r.threshold if T.HIGHER_IS_BETTER[r.metric] else v < r.threshold
        total += ok
    return pd.Series(total / cfg.use_case.nunique(), index=d.index)


rows = []
for q in LEVELS:
    X = df[[f"{m}_p{q}" for m in T.METRICS]].set_axis(list(T.METRICS), axis=1)
    for name, y in [
        ("IQB", contrib[q].sum(axis=1)),
        ("weakest-link variant", weakest_link_iqb(df, q)),
    ]:
        full, add = additive_gap(X, y)
        rows.append(
            {
                "score": name,
                "level": f"P{q}",
                "R² with interactions": full,
                "R² additive only": add,
                "share of explained variance due to interactions": max(
                    0.0, (full - add) / full
                ),
            }
        )
inter = pd.DataFrame(rows)
inter.pivot(
    index="percentile",
    columns=" ",
    values="share of explained variance due to interactions",
).round(3)


Each value is the share of the explained variance that needs interactions between metrics: (R² with interactions − R² additive only) ÷ R² with interactions, from gradient-boosted trees fitted with and without interactions allowed.

A value of 0 means that a model adding up one curve per metric does as well as one that can combine metrics; 0.774 means that 77% of what the model explains at P25 depends on combinations of metrics. 

For IQB as designed, every value is 0.001 or less. For the weakest-link variant, the share falls as the percentile rises, from 77% at P25 to 5% at P95.

In IQB as currently designed, combinations of metrics do not matter: each metric adds its own, separate contribution, and the additive model fits as well as the unconstrained one. 

A use-case score is a weighted average of its requirements, so a failed requirement only subtracts its weight. Excellent download can therefore make up for poor latency in the video-conferencing score, although a video call with poor latency is poor whatever the download speed.

In the weakest-link variant, interactions carry a large share of the explained variance at P25 and P50 (77% and 23%) and less at higher percentiles (10% and 5%). If the intent is that some use cases need several metrics at once, it is the aggregation, not the weights, that has to change.

## Weight contributions

Following Becker et al. (2017), we look for metric weights under which each metric's effective share equals a target share. 

Each metric's tests are scaled by a common factor, and the score is renormalized to the 0–1 scale. 

We use two targets: **as intended**, where each metric's effective share equals its nominal share (download 27%, upload 19%, latency 26%, loss 28%), and **equal**, where each metric explains a quarter.

A metric whose contribution is (almost) constant at a percentile, shown as "n/a", and the other metrics' targets are rescaled among themselves.

In [ ]:
def calibrate(c, target):
    cm = c.to_numpy()
    nom = nominal.reindex(c.columns).to_numpy()
    active = np.array([pd.Series(cm[:, j]).value_counts(normalize=True).iloc[0] < 0.99 for j in range(cm.shape[1])])

    def shares(a):
        y = cm @ a
        return np.array([np.cov(cm[:, j] * a[j], y)[0, 1] for j in range(cm.shape[1])]) / y.var()

    def loss(z):
        a = np.zeros(len(nom))
        a[active] = np.exp(z)
        t = target[active] / target[active].sum()
        return ((shares(a)[active] - t) ** 2).sum()

    res = minimize(loss, np.zeros(active.sum()), method="Nelder-Mead",
                   options={"xatol": 1e-6, "fatol": 1e-10, "maxiter": 4000})
    a = np.ones(len(nom))
    a[active] = np.exp(res.x)  # constant metrics keep their current weight
    weights = a * nom / (a * nom).sum()  # metric weights after rescaling, summing to 1
    out = pd.Series(weights, index=c.columns)
    out[~active] = np.nan
    return out, shares(a)


rows = {}
for tname, target in [("as intended", nominal_share.to_numpy()), ("equal", np.full(4, 0.25))]:
    for q in LEVELS:
        w_new, s_new = calibrate(contrib[q], target)
        rows[(tname, f"P{q}")] = w_new
calib = pd.DataFrame(rows).reindex(list(T.METRICS))
calib.insert(0, ("current", "all"), nominal_share)
print("Metric weights (sum to 1) that make the effective shares match the target; NaN = unreachable at that percentile")
calib.round(2)

Each column is a set of metric weights that add up to 1. 

The first is the current weighting. 

The next four give, for each percentile, the weights under which each metric's effective share would equal its nominal share (*as intended*), and the last four the weights under which every metric would explain a quarter (*equal*). 

NaN marks a metric that is constant at that percentile (one value in at least 99% of country-months), so that no weight can give it influence; here, that is packet loss at P95.

To read a column, compare it with the current weights. For example, for the effective shares at P25 to match the nominal ones, upload would need 0.31 instead of 0.19 and packet loss 0.41 instead of 0.28, with download and latency at 0.14 each. The needed weights change a lot from one percentile to the next, as the figure and the paragraph below show.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8), sharey=True)
slot, bar = 0.12, 0.11
for ax, tname in zip(axes, ["as intended", "equal"]):
    cols = [f"P{q}" for q in LEVELS]
    x = np.arange(len(cols))
    for k, m in enumerate(T.METRICS):
        vals = calib.loc[m, [(tname, c) for c in cols]].to_numpy(dtype=float)
        ax.bar(x + (k - 1.5) * slot, np.nan_to_num(vals), bar, color=METRIC_COLOR[m], label=METRIC_LABEL[m])
        for xi, v in zip(x, vals):
            if np.isnan(v):
                ax.text(xi + (k - 1.5) * slot, 0.01, "n/a", ha="center", va="bottom", fontsize=7,
                        color=T.MUTED, rotation=90)
        ax.hlines(nominal_share[m], x.min() - 0.5, x.max() + 0.5, color=METRIC_COLOR[m], lw=0.8, alpha=0.5)
    ax.set_xticks(x, cols)
    ax.set_title(f"target: {tname}", fontsize=10)
    ax.grid(axis="x", visible=False)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
axes[0].set_ylabel("metric weight needed")
fig.suptitle("Weights needed for each metric's actual influence to match the target (thin lines: current weights)",
             x=0.01, ha="left", fontweight="bold")
fig.tight_layout(rect=(0, 0.07, 1, 0.95))
fig.legend(*axes[0].get_legend_handles_labels(), ncol=4, fontsize=9, loc="lower center", bbox_to_anchor=(0.5, 0.0))
T.savefig(fig, "2b_calibrated_weights")

The figure shows the table as bars, with one panel for each target. 

In each panel, the x-axis is the percentile and each group of four bars gives the metric weights needed there; the y-axis is the weight, as a share that adds up to 100% within each group. 

The thin horizontal line in each metric's color is its current weight, so a bar above its line means the metric would need more weight than it has now, and a bar below it means less. *n/a* marks packet loss at P95, which no weight can make matter.

Only at P50 do the bars stay close to their lines. At P25, upload and packet loss rise well above theirs while download and latency fall to about half; at P75 packet loss rises to about 45%; and at P95 download rises to about 40%. The two targets give similar pictures.

To give each metric its intended influence, the weights would have to change with the percentile. 

At **P50** small changes would do: download 0.24, upload 0.23, latency 0.30 and loss 0.24, instead of 0.27, 0.19, 0.26 and 0.28, so the current weights are roughly right for the median. 

At **P25** upload and loss would need far more weight (0.31 and 0.41), and download and latency about half of theirs. 

At **P75** loss would need 0.45. 

At **P95** packet loss cannot be made to matter at any weight, because every country passes, and download would need 0.43, because it almost always passes too. The equal-importance target tells the same story.

No single set of weights therefore delivers the intended influence across percentiles. This follows from the step thresholds: a metric influences the score only where its thresholds cut through the distribution, and where that happens moves with the percentile.

## How much do the metrics overlap?

The four metrics are not independent measurements. Highly correlated metrics partly double-count the same underlying property, here network capacity.


In [ ]:
X50 = df[[f"{m}_p50" for m in T.METRICS]].set_axis(
    [METRIC_LABEL[m] for m in T.METRICS], axis=1
)
rho = pd.DataFrame(spearmanr(X50).statistic, index=X50.columns, columns=X50.columns)
ranks = X50.rank().to_numpy()
ranks = (ranks - ranks.mean(0)) / ranks.std(0)
eig = np.linalg.eigvalsh(np.corrcoef(ranks.T))[::-1]
print(
    "Share of the joint variation of the four metrics (ranks, P50) captured by each principal component:",
    (eig / eig.sum()).round(2),
)
rho.round(2)


The table gives the Spearman rank correlation between each pair of metrics at P50, across the 3,834 country-months: 1 means two metrics rank country-months in exactly the same order. 

Latency and packet loss correlate negatively with download and upload because lower latency and loss are better, so a negative value here still means that networks which are better on one metric tend to be better on the other.

The printed line splits the joint variation of the four metrics into independent components (the principal components of their ranks). The first captures 70%, the next two 15% and 12%, and the last 3%.

The first principal component captures 70% of the joint variation of the four metrics; download and upload alone correlate at ρ = 0.86. 

The metrics largely measure one thing, overall connection quality, plus smaller latency and loss dimensions (15% and 12%). Giving download and upload 46% of the nominal weight between them therefore weights capacity heavily: equal weights on correlated metrics are not equal weights on independent dimensions.


## Save

In [ ]:
eff.to_csv(T.TAB / "metric_effective_shares.csv")
rf.to_csv(T.TAB / "metric_random_forest.csv", index=False)
inter.to_csv(T.TAB / "metric_interactions.csv", index=False)
calib.to_csv(T.TAB / "metric_calibrated_weights.csv")
print("saved metric_effective_shares, metric_random_forest, metric_interactions and metric_calibrated_weights to", T.TAB)

## Summary

### Which metric makes a difference?

It depends on the percentile, because a metric only moves the score where its thresholds cut through the distribution:

| | most influential | least influential |
|---|---|---|
| P25 | download, latency | upload, packet loss |
| P50 | packet loss, download | upload |
| P75 | latency, upload, download | packet loss |
| P95 | latency, upload | packet loss (none at all) |

### Are the weights appropriate?

Internally, only approximately and only at P50. Elsewhere the effective weighting diverges strongly from the intended one, and at P95 packet loss's 28% weight is a constant. Combinations of metrics never matter, because the formula is additive. Whether the *intended* weights are themselves right cannot be answered from M-Lab data alone: that needs an outcome outside the score.

### What should change?

1. **Report effective weights next to nominal ones** for whichever percentile is published. It can serve as a safeguard against a weight that does nothing (e.g. loss at P95) or more than intended.
2. **If the intended importance matters, calibrate per percentile** (the calibration section above, after Becker et al. 2017), or publish a single percentile and calibrate.
3. **Better: replace step thresholds with ramps.** The framework already defines a minimum and a high-quality threshold for each requirement. Scoring each requirement on a sliding scale between the two (log scale for throughput and latency) would let every metric influence the score across its whole range. It would also make effective weights much less dependent on the percentile.
4. **Decide whether use cases should be compensatory.** Today a failed requirement costs only its weight, so good download can make up for poor latency. If a use case is unusable when any one requirement fails, score it with a weakest-link (minimum) or geometric-mean rule. The combinations section shows this introduces the metric combinations the current formula lacks.
5. **Weight dimensions, not correlated metrics.** For example, *capacity* (download and upload) and *responsiveness* (latency and loss), so that correlated metrics are not double-counted.
6. **To learn appropriate weights, use an external outcome.** The right job for a random forest, or gradient boosting with SHAP values, is to predict a measure of *user experience* from the four metrics: application-level quality (video start-up and rebuffering, call quality), per-use-case scores from an independent speed test (for example Cloudflare's AIM scores), or survey-based satisfaction. Its importances and interactions would then be evidence about what the weights *should* be, rather than a reflection of the weights already chosen.